In [ ]:
# K means cluster on the textured faces of an obj file.
# Sample texture colors at face centers as an average of the vertex values and cluster based on HSV values, not RGB.

from cv2 import cvtColor, COLOR_BGR2HSV
import numpy as np
from sklearn.cluster import KMeans
import trimesh
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib import cm
import os

# Load mesh
path = "FILEPATH" 
filename = "FILENAME.obj"
obj_path = os.path.join(path, filename)

# Load the textured OBJ file using trimesh
print(f"Loading mesh {obj_path}...")
mesh = trimesh.load(obj_path, force='mesh')
vertices = mesh.vertices 
faces = mesh.faces
print(f"Loaded mesh with {len(vertices)} vertices and {len(faces)} faces")

In [ ]:
# Check for UV coordinates and texture image

print("Checking for UV coordinates and texture image...")
if hasattr(mesh.visual, 'uv') and mesh.visual.uv is not None: 
    uv_coords = mesh.visual.uv
else:
    raise ValueError("No UV coordinates found")

if hasattr(mesh.visual, 'material') and hasattr(mesh.visual.material, 'image'):
    texture_image = mesh.visual.material.image
else:
    raise ValueError("No texture image found in material")

texture_np = np.array(texture_image.convert('RGB')) 
print("Texture loaded")

# Sample face color at face center UVs
print("Sampling texture colors at face centers...")
face_uvs = uv_coords[faces] 
face_uvs_avg = face_uvs.mean(axis=1) 

# Flip V axis of the UV map to match image coordinates in the numpy array. To avoid distorted projected texture on the 3D mesh.
face_uvs_avg[:, 1] = 1.0 - face_uvs_avg[:, 1] 

# Sample the texture image to extract the colour of each face in the mesh.
uv_px = np.round(face_uvs_avg * [texture_np.shape[1] - 1, texture_np.shape[0] - 1]).astype(int)
uv_px[:, 0] = np.clip(uv_px[:, 0], 0, texture_np.shape[1] - 1) 
uv_px[:, 1] = np.clip(uv_px[:, 1], 0, texture_np.shape[0] - 1) 
face_colors_rgb = texture_np[uv_px[:, 1], uv_px[:, 0]] 
face_colors_bgr = face_colors_rgb[..., ::-1] 
face_colors_hsv = cvtColor(face_colors_bgr.reshape(-1, 1, 3), COLOR_BGR2HSV).reshape(-1, 3)

print("Done")

In [ ]:
# K-means clustering
print("Performing K-means clustering on face colors...")

# adjust the number of k clusters as needed
k = 8 

# Select which HSV channels to use: 0=Hue, 1=Saturation, 2=Value
channels = [1,2] 

# k means with k clusters, fixed random state for reproducibility 
kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto') 
face_labels = kmeans.fit_predict(face_colors_hsv[:, channels].astype(np.float32))

print("K-Means Done")

In [ ]:
# Elbow plot to determine optimal number of clusters
print("Computing elbow plot...")

inertias = []
k_range = range(2, 11)

for k_test in k_range:
    kmeans_test = KMeans(n_clusters=k_test, random_state=42, n_init='auto')
    kmeans_test.fit(face_colors_hsv[:,channels].astype(np.float32))
    inertias.append(kmeans_test.inertia_)
    print(f"k={k_test}: inertia={kmeans_test.inertia_:.2f}")

# Plot the elbow curve
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
plt.title('Elbow Plot for K-Means Clustering', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(k_range)

os.makedirs('results', exist_ok=True)
plt.savefig('results/elbow_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("Elbow plot saved to results/elbow_plot.png")

In [ ]:
# Helper function to set equal aspect ratio for 3D plots to avoid distortion of plotted meshes on the 3d plot grid.

def set_axes_equal(ax, vertices):
    x_limits = [vertices[:, 0].min(), vertices[:, 0].max()]
    y_limits = [vertices[:, 1].min(), vertices[:, 1].max()]
    z_limits = [vertices[:, 2].min(), vertices[:, 2].max()]
    
    max_range = max(
        x_limits[1] - x_limits[0],
        y_limits[1] - y_limits[0],
        z_limits[1] - z_limits[0]
    ) / 2.0
    
    mid_x, mid_y, mid_z = np.mean(x_limits), np.mean(y_limits), np.mean(z_limits)
    
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range * 1.2, mid_y + max_range * 1.2)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)

In [ ]:
# Generate distinct colors for each cluster using a colormap
cmap = cm.get_cmap('tab20', k)   
colors = [cmap(i) for i in range(k)]

In [ ]:
# Plot clusters on mesh
print("Plotting clusters on mesh...")

# Create a downsample of face_centers and corresponding labels/colors for faster plotting
face_centers = mesh.triangles_center 
sample_size_centers = int(len(face_centers) * 0.7) # change value here to define downsample size
sample_indices_centers = np.random.choice(len(face_centers), size=sample_size_centers, replace=False)
face_centers_sample = face_centers[sample_indices_centers]
face_colors_rgb_sample = face_colors_rgb[sample_indices_centers]
face_labels_sample_centers = face_labels[sample_indices_centers]

# First subplot
fig = plt.figure(figsize=(20, 10))
ax = fig.add_subplot(121, projection='3d') # 1 row, 1 colunm, first subplot at position 1 with a 3d projection style
ax.set_title("Original Textured Mesh")
ax.scatter(face_centers_sample[:, 0], face_centers_sample[:, 1], face_centers_sample[:, 2], c=face_colors_rgb_sample / 255.0, s=0.2)
set_axes_equal(ax, vertices) 
ax.view_init(elev=90, azim=0) 

# Second subplot
ax2 = fig.add_subplot(122, projection='3d') 
ax2.set_title("Face Cluster Mapping")
sc2 = ax2.scatter(
    face_centers_sample[:, 0], face_centers_sample[:, 1], face_centers_sample[:, 2],
    c=face_labels_sample_centers, cmap=cmap, s=0.2
) 

set_axes_equal(ax2, vertices) 
ax2.view_init(elev=90, azim=0)

# Legend setup
counts = np.bincount(face_labels_sample_centers, minlength=k) 
handles = [
    Line2D([0], [0], 
           marker='o', linestyle='', markersize=8,
           markerfacecolor=colors[i], markeredgecolor='k', 
           label=f'Cluster {i} (n={counts[i]})')
    for i in range(k)
]
ax2.legend(handles=handles, title="Clusters",
           loc='upper left', bbox_to_anchor=(1.02, 1), frameon=True) 

os.makedirs('results', exist_ok=True)
plt.savefig('results/side_by_side_kmeans.png', dpi=300, bbox_inches='tight')
plt.tight_layout() 
plt.show()

In [ ]:
# Calculate average saturation and brightness value for each cluster
import pandas as pd

print("\nAverage Saturation and Brightness for each cluster:")

brightness_saturation_stats = []

for i in range(k):
    cluster_mask = (face_labels == i)
    avg_saturation = np.mean(face_colors_hsv[cluster_mask, 1])
    avg_brightness = np.mean(face_colors_hsv[cluster_mask, 2])
    brightness_saturation_stats.append((i, avg_saturation, avg_brightness))
    print(f"Cluster {i}: Avg Saturation = {avg_saturation:.2f}, Avg Brightness = {avg_brightness:.2f}")
    
summary = pd.DataFrame.from_records(brightness_saturation_stats, columns=['Cluster', 'Avg_Saturation', 'Avg_Brightness'])

# Export to CSV
os.makedirs('results', exist_ok=True)
summary.to_csv('results/cluster_brightness_saturation_stats.csv', index=False, sep=';', decimal=',')

In [ ]:
# Plot average saturation and brightness as a cross plot

plt.figure(figsize=(8, 6))
plt.scatter(summary['Avg_Saturation'], summary['Avg_Brightness'], c=summary['Cluster'], cmap=cmap, s=100)

for i, row in summary.iterrows():
    plt.text(row['Avg_Saturation'], row['Avg_Brightness'], str(int(row['Cluster'])), fontsize=9, ha='center', va='center', color='white')

plt.xlabel('Average Saturation', fontsize=12)
plt.ylabel('Average Brightness', fontsize=12)
plt.title(f'Average Saturation vs Brightness KBN-AV6', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

# Save the saturation vs brightness plot
os.makedirs('results', exist_ok=True)
plt.savefig('results/avg_saturation_brightness.png', dpi=300)

In [ ]:
# Select damage clusters manually

print("\nCluster stats:")
for i in range(k): 
    count = np.sum(face_labels == i) 
    print(f"Cluster {i}: {count} faces")

damage_clusters = [6]  # Update selected cluster here
print(f"\nSelected damage clusters: {damage_clusters}")

# Create buffer around selected damage clusters
mesh_full = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)  
adj = mesh_full.face_adjacency
n_faces = len(faces) 

# Calculate average edge length and convert to metric buffer
edge_lengths = mesh_full.edges_unique_length  
avg_edge_length = np.mean(edge_lengths)  
buffer_distance_mm = 0.5 # Define desired buffer distance in mm
buffer_distance_m = buffer_distance_mm / 1000.0  
n_rings = int(np.ceil(buffer_distance_m / avg_edge_length)) 
print(f"\nBuffer calculation:")
print(f"  Average edge length: {avg_edge_length*1000:.3f} mm")
print(f"  Target buffer distance: {buffer_distance_mm} mm")
print(f"  Calculated n_rings: {n_rings}")

# Faces in selected damage clusters
damage_mask = np.isin(face_labels, damage_clusters)  
damage_with_buffer_mask = damage_mask.copy()  
frontier_mask = damage_mask.copy() 

for _ in range(n_rings):  
    mask0 = frontier_mask[adj[:, 0]]  
    mask1 = frontier_mask[adj[:, 1]]  
    neighbors0 = adj[mask0, 1]  
    neighbors1 = adj[mask1, 0]  
    neighbors = np.concatenate([neighbors0, neighbors1])  
    neighbors = np.unique(neighbors)  

    # Keep only faces not already marked as damage/buffer
    neighbors = neighbors[~damage_with_buffer_mask[neighbors]]

    if neighbors.size == 0:  
        break

    damage_with_buffer_mask[neighbors] = True  

    frontier_mask = np.zeros(n_faces, dtype=bool)  
    frontier_mask[neighbors] = True

# Boolean mask for faces not in the selected damage clusters (with buffer)
undamaged_faces_mask = ~damage_with_buffer_mask  
undamaged_faces = faces[undamaged_faces_mask] 

# Remap vertices and faces for clean export
unique_vertices, inverse_indices = np.unique(undamaged_faces.reshape(-1), return_inverse=True) 
undamaged_vertices = vertices[unique_vertices] 
undamaged_faces_remapped = inverse_indices.reshape(-1, 3) 

# Remap face colors to match the undamaged faces
undamaged_face_colors = face_colors_rgb[undamaged_faces_mask]

# Create new mesh without the damage that was selected in the damage clusters line
undamaged_mesh = trimesh.Trimesh(
    vertices=undamaged_vertices, 
    faces=undamaged_faces_remapped, 
    face_colors=undamaged_face_colors,
    process=False
)

# Create mesh containing only the selected damage clusters (core + N-ring buffer)
damaged_faces_mask = damage_with_buffer_mask  
damaged_faces = faces[damaged_faces_mask]

# Remap vertices and faces for clean export
unique_vertices_damaged, inverse_indices_damaged = np.unique(damaged_faces.reshape(-1), return_inverse=True)
damaged_vertices = vertices[unique_vertices_damaged]
damaged_faces_remapped = inverse_indices_damaged.reshape(-1, 3)

# Remap face colors to match the damaged faces
damaged_face_colors = face_colors_rgb[damaged_faces_mask]

# Create new mesh with only the damage clusters
damaged_mesh = trimesh.Trimesh(
    vertices=damaged_vertices,
    faces=damaged_faces_remapped,
    face_colors=damaged_face_colors,
    process=False
)

# Export
damaged_path = obj_path.replace('.obj', f'_damaged_cluster_buffered.obj')
damaged_mesh.export(damaged_path)
print(f"Damaged mesh exported to: {damaged_path}")